# Thư viện OpenCV

In [ ]:
! pip install opencv-python
! pip install paddleocr==2.8.1

In [1]:
import numpy as np

In [ ]:
import cv2
import os

ROOT = "data/input"
file_name = "deskew.jpg"

path = os.path.join(ROOT, file_name)

In [ ]:
def deskew_image(image_path, output_path=None, max_angle=45):
    """
    Args:
        image_path (str): Đường dẫn đến ảnh đầu vào
        output_path (str, optional): Đường dẫn lưu ảnh sau khi deskew
        max_angle (int): Giới hạn góc tối đa (± độ)
    
    Returns:
        img_deskewed: Ảnh đã được sửa góc
    """
    # Đọc ảnh
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Không đọc được ảnh: {image_path}")
    
    # Chuyển sang grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Tăng độ tương phản + nhị phân
    gray = cv2.bitwise_not(gray)
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
    
    # Tìm các contours
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Lấy contour lớn nhất 
    if not contours:
        return img  # Không tìm thấy contour nào
    
    largest_contour = max(contours, key=cv2.contourArea)
    
    # Tính góc nghiêng từ rotated rectangle
    rect = cv2.minAreaRect(largest_contour)
    angle = rect[-1]
    
    # Chuẩn hóa góc
    if angle < -45:
        angle = 90 + angle
    elif angle > 45:
        angle = angle - 90
    
    # Giới hạn góc
    angle = max(min(angle, max_angle), -max_angle)
    
    print(f"Góc nghiêng phát hiện: {angle:.2f}°")
    
    # Tạo ma trận xoay
    (h, w) = img.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    
    # Xoay ảnh
    rotated = cv2.warpAffine(img, M, (w, h), 
                             flags=cv2.INTER_CUBIC, 
                             borderMode=cv2.BORDER_REPLICATE)
    
    return rotated

In [ ]:
#auto-crop (cắt viền thừa)
_, thresh = cv2.threshold(
    deskewed, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

coords = cv2.findNonZero(thresh)

if coords is not None:
    x, y, w, h = cv2.boundingRect(coords)
    cropped = deskewed[y:y+h, x:x+w]
else:
    # fallback: giữ nguyên ảnh
    cropped = deskewed

if len(cropped.shape) == 3:
    cropped = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)

In [ ]:
#Xử lý nền: giấy ố vàng, nền không đều + không làm với chữ màu
normalized = cv2.adaptiveThreshold(cropped, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)

In [ ]:
#Làm rõ chữ sharpen
kernel = np.array([[0, -1, 0],
                   [-1, 5,-1],
                   [0, -1, 0]])
sharpened = cv2.filter2D(normalized, -1, kernel)